# 18D — Consolidated Fannie multi-year dataset construction
Process the current-release quarterly archives once, retaining the lowest deterministic loan hashes for each origination year in each archive. Merge these candidates into exact annual samples. Apply Notebook 15 rules without fitting any model. Checkpoints contain local loan-level information; never commit them or upload them. The population is bounded to acquisitions during the origination year and following calendar year. This is not a complete census of all originations.

In [1]:
from pathlib import Path
import hashlib, heapq, json, re, sqlite3, time, zipfile, shutil
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW = ROOT / 'data/raw/Fannie_Mae/release_2026Q2'
OUT = ROOT / 'data/processed/Fannie_Mae/notebook_18d_consolidated_v1'
OUT.mkdir(parents=True, exist_ok=True)
YEARS = list(range(2015, 2024))
K = 50_000
SEED = 42
FILES = [f'{y}Q{q}' for y in range(2015, 2025) for q in range(1, 5)]
VERSION = '18d-v1-seed42-k50000'
# Process available files now; final export requires all 40.
available = [f for f in FILES if (RAW / f'{f}.zip').is_file()]
print('Available:', len(available), '/', len(FILES))
print('Missing:', ', '.join(f for f in FILES if f not in available))
print('Free GiB:', round(shutil.disk_usage(OUT).free / 2**30, 1))
assert available, 'No quarterly ZIP files found.'


Available: 27 / 40
Missing: 2015Q1, 2015Q2, 2015Q3, 2015Q4, 2016Q1, 2016Q2, 2016Q3, 2016Q4, 2017Q1, 2017Q2, 2017Q3, 2017Q4, 2024Q3
Free GiB: 433.4


## 1. Reuse the reviewed field positions and target rules
The current glossary confirms positions 2–73 used below and position 111 for Origination Classic FICO. The primary specification excludes FICO. Legacy minimum-disclosed FICO and field 111 are retained separately for a later sensitivity decision; they are not interchangeable. Physical schema checks and code checks stop on unexpected values.

In [2]:
POSITIONS = {'loan_id':2,'report':3,'channel':4,'rate':8,'upb':10,'term':13,'orig':14,'first_payment':15,'ltv':20,'cltv':21,'borrowers':22,'dti':23,'borrower_fico':24,'coborrower_fico':25,'purpose':27,'property_type':28,'units':29,'occupancy':30,'state':31,'mi':34,'amortization':35,'delinq':40,'zero_code':44,'zero_month':45,'mi_type':73,'classic_fico':111,'vantage_score':114}
ADVERSE = {'02','03','09','15'}
KNOWN = ADVERSE | {'01','06','16','96'}
DATE = re.compile(r'(0[1-9]|1[0-2])(19|20)\d{2}')
def month(v):
    return int(v[2:])*12+int(v[:2])-1 if DATE.fullmatch(v) else None

def target_for(rows):
    orig = {month(r['orig']) for r in rows}
    fp = {month(r['first_payment']) for r in rows}
    result = dict(quarter=None, primary=False, strict=False, serious=False, adverse=False, reason='unresolved_dates', missing_months=None, unknown_active=False, repurchase=False, terminal_timing_disagreements=0)
    reports = [month(r['report']) for r in rows]
    if None in reports or len(set(reports)) != len(reports):
        raise ValueError('Invalid or duplicate reporting month; stop and investigate.')
    if any(r['zero_code'] not in KNOWN|{''} for r in rows):
        raise ValueError('Unexpected terminal code; stop and investigate.')
    if any(not (re.fullmatch(r'\d{1,2}',r['delinq']) or r['delinq'] in {'XX',''}) for r in rows):
        raise ValueError('Unexpected delinquency code; stop and investigate.')
    terminal = [(r,m) for r,m in zip(rows,reports) if r['zero_code']]
    if len({r['zero_code'] for r,m in terminal})>1 or len({r['zero_month'] for r,m in terminal})>1:
        raise ValueError('Conflicting terminal records; review before export.')
    if any(month(r['zero_month']) is None for r,m in terminal):
        raise ValueError('Terminal code without valid effective month.')
    if len(orig)!=1 or len(fp)!=1 or None in orig or None in fp:
        return result
    o=next(iter(orig)); start=next(iter(fp)); end=start+23
    result['quarter']=f'{o//12}Q{(o%12)//3+1}'
    w=[(r,m) for r,m in zip(rows,reports) if start<=m<=end]
    tr=[(r,m) for r,m in w if r['zero_code']]
    left=min(reports)>start
    complete=max(reports)>=end or bool(tr)
    endpoint=min(end,min(m for r,m in tr)) if tr else end
    missing=len(set(range(start,endpoint+1))-{m for r,m in w})
    unknown=any(not r['delinq'].isdigit() and not r['zero_code'] for r,m in w)
    repurchase=any(r['zero_code']=='06' for r,m in tr)
    serious=any(r['delinq'].isdigit() and int(r['delinq'])>=3 for r,m in w)
    adverse=any(r['zero_code'] in ADVERSE for r,m in tr)
    primary=not left and complete
    result.update(primary=primary,strict=primary and missing==0 and not unknown and not repurchase, serious=serious,adverse=adverse,reason='left_truncated' if left else ('incomplete_followup' if not complete else 'eligible'),missing_months=missing,unknown_active=unknown,repurchase=repurchase,terminal_timing_disagreements=sum((start<=m<=end)!=(start<=month(r['zero_month'])<=end) for r,m in terminal))
    return result

def demo(offsets,event=None,terminal=None,unknown=None):
    start=2018*12+2
    def fmt(m): return f'{m%12+1:02d}{m//12:04d}'
    return [dict(orig='012018',first_payment='032018',report=fmt(start+i),delinq='XX' if i==unknown else ('03' if i==event else '00'),zero_code=terminal[1] if terminal and i==terminal[0] else '',zero_month=fmt(start+i) if terminal and i==terminal[0] else '') for i in offsets]
for rows,expected in [(demo(range(24)),(True,True,False)),(demo(range(24),23),(True,True,True)),(demo(range(25),24),(True,True,False)),(demo(range(7),terminal=(6,'09'),unknown=6),(True,True,True)),(demo(range(7),terminal=(6,'06'),unknown=6),(True,False,False)),(demo(range(1,24)),(False,False,False)),(demo(range(12)),(False,False,False)),(demo([i for i in range(24) if i!=5]),(True,False,False)),(demo(range(24),unknown=5),(True,False,False))]:
    r=target_for(rows)
    assert (r['primary'],r['strict'],r['serious'] or r['adverse'])==expected
print('Nine Notebook 15 target tests passed.')


Nine Notebook 15 target tests passed.


## 2. Scan and checkpoint each archive
For every archive, store its loan IDs for the duplicate audit and retain at most 50,000 lowest-hash candidates per qualifying origination year. Selecting the global lowest 50,000 from the union of local lowest 50,000 is exact when IDs are unique. No label or eligibility is used for ranking. Unresolved origination dates cannot be assigned to an annual population and are counted separately. A target-rule exception is retained as a diagnostic and blocks final export if the affected loan enters the final sample. Checkpoints are only reused when the compressed-file hash and settings match.

In [3]:
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
    return h.hexdigest()

def rank(key):
    return int.from_bytes(hashlib.blake2b(key.encode('ascii'),digest_size=8,person=b'Fannie18-seed42').digest(),'big')

CONFLICT_FIELDS = ['rate','upb','term','ltv','cltv','dti','borrower_fico','coborrower_fico','mi','mi_type','purpose','channel','units','occupancy','state','property_type','amortization','borrowers']

def scan_archive(tag):
    path=RAW/f'{tag}.zip'; digest=sha(path); folder=OUT/tag; folder.mkdir(exist_ok=True)
    marker=folder/'complete.json'; candidates=folder/'candidates.parquet'; idsfile=folder/'ids.parquet'
    if marker.exists() and candidates.exists() and idsfile.exists():
        meta=json.loads(marker.read_text())
        if meta['version']==VERSION and meta['source_sha256']==digest and meta['candidate_sha256']==sha(candidates) and meta['ids_sha256']==sha(idsfile):
            print(tag, 'checkpoint reused', flush=True); return meta
        raise ValueError(f'{tag}: checkpoint changed; preserve old run and use a new OUT folder.')
    begin=time.perf_counter(); heaps=defaultdict(list); seen=set(); counts=Counter(); rows_count=0
    current=None; history=[]; acquisition_year=int(tag[:4]); code_errors=Counter(); field_profiles=Counter()
    def finish(key, rawrows):
        if key is None: return
        if key in seen: raise ValueError(f'{tag}: noncontiguous loan history')
        seen.add(key)
        dates={month(p[13].strip().decode('ascii')) for p in rawrows}
        if len(dates)!=1 or None in dates:
            counts['unresolved']+=1; return
        o=next(iter(dates)); y=o//12; q=f'{y}Q{(o%12)//3+1}'
        counts[q]+=1
        if y not in YEARS or acquisition_year not in (y,y+1): return
        counts[f'frame_{y}']+=1
        h=rank(key); heap=heaps[y]
        if len(heap)>=K and h>=-heap[0][0]: return
        records=[{name:p[pos-1].strip().decode('ascii') for name,pos in POSITIONS.items()} for p in rawrows]
        first=min(records,key=lambda r:month(r['report']) if month(r['report']) is not None else -1)
        error=''
        try: target=target_for(records)
        except ValueError as exc:
            error=str(exc); code_errors[error]+=1
            target=dict(quarter=q,primary=False,strict=False,serious=False,adverse=False,reason='target_rule_error',missing_months=None,unknown_active=False,repurchase=False,terminal_timing_disagreements=0)
        row={**first,**target,'origination_year':y,'ORIGINATION_COHORT':q,'acquisition_file':tag,'rank_hex':f'{h:016x}','target_error':error,'history_rows':len(records),'origination_field_conflict_count':sum(len({r[k] for r in records if r[k]})>1 for k in CONFLICT_FIELDS)}
        item=(-h,key,row)
        if len(heap)<K: heapq.heappush(heap,item)
        else: heapq.heapreplace(heap,item)
    with zipfile.ZipFile(path) as z:
        members=[m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.csv')]
        assert len(members)==1, f'{tag}: expected one CSV'
        with z.open(members[0]) as f:
            for rows_count,line in enumerate(f,1):
                parts=line.rstrip(b'\r\n').split(b'|')
                if len(parts)!=114: raise ValueError(f'{tag}: unexpected width at row {rows_count}')
                key=parts[1].strip().decode('ascii')
                if not re.fullmatch(r'[A-Za-z0-9]{1,12}',key): raise ValueError(f'{tag}: invalid ID')
                if not DATE.fullmatch(parts[2].strip().decode('ascii')): raise ValueError(f'{tag}: invalid reporting month')
                if key!=current:
                    finish(current,history); current=key; history=[]
                history.append(parts)
                if rows_count%5_000_000==0: print(tag, f'{rows_count:,} rows',flush=True)
            finish(current,history)
    assert sha(path)==digest, 'Source changed during scan'
    data=pd.DataFrame([item[2] for heap in heaps.values() for item in heap])
    if data.empty: data=pd.DataFrame(columns=['loan_id','origination_year','rank_hex'])
    data.to_parquet(candidates,index=False)
    pd.DataFrame({'loan_id':sorted(seen)}).to_parquet(idsfile,index=False)
    meta=dict(version=VERSION,archive=tag,source_sha256=digest,source_bytes=path.stat().st_size,rows=rows_count,loans=len(seen),counts=dict(counts),candidate_count=len(data),candidate_target_errors=dict(code_errors),seconds=time.perf_counter()-begin,candidate_sha256=sha(candidates),ids_sha256=sha(idsfile))
    tmp=folder/'complete.tmp'; tmp.write_text(json.dumps(meta,indent=2)); tmp.replace(marker)
    print(tag, 'completed', len(seen), 'loans;', len(data), 'candidates',flush=True)
    return meta


In [4]:
metadata=[scan_archive(tag) for tag in available]
print('Completed archives:', len(metadata), '/',len(FILES))


2018Q1 5,000,000 rows
2018Q1 10,000,000 rows
2018Q1 15,000,000 rows
2018Q1 20,000,000 rows
2018Q1 completed 458830 loans; 100000 candidates
2018Q2 5,000,000 rows
2018Q2 10,000,000 rows
2018Q2 15,000,000 rows
2018Q2 20,000,000 rows
2018Q2 completed 454581 loans; 52493 candidates
2018Q3 5,000,000 rows
2018Q3 10,000,000 rows
2018Q3 15,000,000 rows
2018Q3 20,000,000 rows
2018Q3 completed 500018 loans; 50310 candidates
2018Q4 5,000,000 rows
2018Q4 10,000,000 rows
2018Q4 15,000,000 rows
2018Q4 completed 421040 loans; 50171 candidates
2019Q1 5,000,000 rows
2019Q1 10,000,000 rows
2019Q1 completed 341865 loans; 100000 candidates
2019Q2 5,000,000 rows
2019Q2 10,000,000 rows
2019Q2 completed 407389 loans; 53942 candidates
2019Q3 5,000,000 rows
2019Q3 10,000,000 rows
2019Q3 15,000,000 rows
2019Q3 20,000,000 rows
2019Q3 25,000,000 rows
2019Q3 completed 673662 loans; 51373 candidates
2019Q4 5,000,000 rows
2019Q4 10,000,000 rows
2019Q4 15,000,000 rows
2019Q4 20,000,000 rows
2019Q4 25,000,000 rows
201

## 3. Validate the combined sampling frame and select annual samples
All 40 files are required before final sampling. Use a disk-backed ID index so the cross-file duplicate check does not require loading all identifiers into RAM. A duplicate blocks export; no automatic deduplication or silent choice of source is made.

In [5]:
assert set(available)==set(FILES), 'Partial checkpoints saved. Download missing archives, then Run All again.'
import pyarrow.parquet as pq
index_path=OUT/'duplicate_audit.sqlite'
if index_path.exists(): index_path.unlink()
with sqlite3.connect(index_path) as db:
    db.execute('CREATE TABLE ids (id TEXT PRIMARY KEY) WITHOUT ROWID')
    for tag in FILES:
        try:
            for batch in pq.ParquetFile(OUT/tag/'ids.parquet').iter_batches(batch_size=100_000):
                db.executemany('INSERT INTO ids VALUES (?)', ((s,) for s in batch.column(0).to_pylist()))
            db.commit()
        except sqlite3.IntegrityError as exc:
            raise ValueError(f'Cross-archive duplicate detected at {tag}; review before sampling') from exc
        print('ID audit:',tag,flush=True)
    assert db.execute('SELECT COUNT(*) FROM ids').fetchone()[0]==sum(m['loans'] for m in metadata)
# Merge one origination year at a time to bound memory.
parts=[]
for y in YEARS:
    relevant=[f for f in FILES if int(f[:4]) in (y,y+1)]
    frames=[pd.read_parquet(OUT/f/'candidates.parquet',filters=[('origination_year','==',y)]) for f in relevant]
    frame=pd.concat(frames,ignore_index=True).sort_values(['rank_hex','loan_id']).head(K).copy()
    assert len(frame)==K and frame.loan_id.is_unique, f'{y}: insufficient unique candidates'
    parts.append(frame)
    print(y,len(frame))
sample=pd.concat(parts,ignore_index=True)
assert sample.loan_id.is_unique
errors=sample.loc[sample.target_error.ne(''),'target_error'].value_counts()
print('Selected-loan target exceptions:',errors.to_dict())
assert errors.empty, 'Selected loans violate reviewed target rules; stop before final export.'


AssertionError: Partial checkpoints saved. Download missing archives, then Run All again.

In [6]:
print("Required files:", len(FILES))
print("Available files:", len(available))

print("\nMissing files:")
print(sorted(set(FILES) - set(available)))

print("\nUnexpected files:")
print(sorted(set(available) - set(FILES)))

print("\nCheckpoint folder:")
print(OUT)

print("\nSaved checkpoint folders:")
for folder in sorted(OUT.iterdir()):
    if folder.is_dir():
        print(folder.name)

Required files: 40
Available files: 27

Missing files:
['2015Q1', '2015Q2', '2015Q3', '2015Q4', '2016Q1', '2016Q2', '2016Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '2017Q4', '2024Q3']

Unexpected files:
[]

Checkpoint folder:
e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_18d_consolidated_v1

Saved checkpoint folders:
2018Q1
2018Q2
2018Q3
2018Q4
2019Q1
2019Q2
2019Q3
2019Q4
2020Q1
2020Q2
2020Q3
2020Q4
2021Q1
2021Q2
2021Q3
2021Q4
2022Q1
2022Q2
2022Q3
2022Q4
2023Q1
2023Q2
2023Q3
2023Q4
2024Q1
2024Q2
2024Q4


In [7]:
for tag in sorted(available):
    folder = OUT / tag
    files = sorted(p.name for p in folder.iterdir() if p.is_file())
    print(tag, ":", files)

2018Q1 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2018Q2 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2018Q3 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2018Q4 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2019Q1 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2019Q2 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2019Q3 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2019Q4 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2020Q1 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2020Q2 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2020Q3 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2020Q4 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2021Q1 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2021Q2 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2021Q3 : ['candidates.parquet', 'complete.json', 'ids.parquet']
2021Q4 : ['candidates.parquet', 'complet

In [8]:
available = [
    f for f in FILES
    if (RAW / f'{f}.zip').is_file()
]

print("Available:", len(available), "/", len(FILES))
print("Missing:", sorted(set(FILES) - set(available)))

Available: 40 / 40
Missing: []


## 4. Apply the original primary predictor mapping
Use earliest reporting snapshots as in Notebook 15. Preserve all sampled loans, including those excluded by target eligibility. Keep original score fields separately; no revised FICO mapping is asserted. The primary model input uses the 22 predictors below.

In [ ]:
first=sample

def valid_number(s, low, high):
    v=pd.to_numeric(s,errors='coerce'); return v.where(v.between(low,high))
def harmonize_mi(percent,kind):
    v=valid_number(percent,1,55); no_mi=percent.eq('') & kind.eq(''); insured=kind.isin(['1','2','3'])
    inconsistent=~(no_mi | (insured & (percent.eq('') | v.notna())))
    return v.where(insured).mask(no_mi,0.0),inconsistent
mi,mi_bad=harmonize_mi(first.mi,first.mi_type)
x=pd.DataFrame(index=first.index)
dti=valid_number(first.dti,1,998); ltv=valid_number(first.ltv,1,998); cltv=valid_number(first.cltv,1,998)
upb=pd.to_numeric(first.upb,errors='coerce'); upb=upb.where(upb.gt(0))
x['ORIGINAL_DTI']=dti; x['ORIGINAL_LTV']=ltv; x['ORIGINAL_INTEREST_RATE']=valid_number(first.rate,0.01,100); x['LOG_ORIGINAL_UPB']=np.log1p(upb)
x['CLTV_MINUS_LTV']=(cltv-ltv).clip(lower=0); x['HAS_SUBORDINATE_FINANCING']=cltv.gt(ltv).astype('int8'); x['HIGH_LTV_GT_80']=ltv.gt(80).astype('int8')
for name,s in [('ORIGINAL_DTI',dti),('ORIGINAL_LTV',ltv),('CLTV',cltv)]: x[name+'_MISSING']=s.isna().astype('int8')
x['MI_PERCENT']=mi; x['HAS_MORTGAGE_INSURANCE']=mi.gt(0).astype('int8'); x['MI_PERCENT_MISSING']=mi.isna().astype('int8')
for name,field,allowed in [('NUMBER_OF_UNITS','units',{'1','2','3','4'}),('OCCUPANCY_STATUS','occupancy',{'P','S','I'}),('CHANNEL','channel',{'B','C','R'}),('AMORTIZATION_TYPE','amortization',{'FRM','ARM'}),('PROPERTY_TYPE','property_type',{'SF','PU','CO','CP','MH'})]: x[name]=first[field].where(first[field].isin(allowed))
x['PROPERTY_STATE']=first.state.where(first.state.str.fullmatch('[A-Z]{2}'))
x['ORIGINAL_LOAN_TERM']=first.term.where(first.term.str.fullmatch('[0-9]+'))
b=pd.to_numeric(first.borrowers,errors='coerce'); x['NUMBER_OF_BORROWERS_GROUP']=pd.Series(pd.NA,index=first.index,dtype='string').mask(b.eq(1),'1').mask(b.between(2,10),'2+')
x['PURCHASE_REFINANCE_GROUP']=first.purpose.map({'P':'PURCHASE','C':'REFINANCE','R':'REFINANCE'})
assert len(x.columns)==22
legacy=pd.concat([valid_number(first.borrower_fico,300,850),valid_number(first.coborrower_fico,300,850)],axis=1).min(axis=1)
classic=valid_number(first.classic_fico,300,850)
output=pd.concat([first[['loan_id','orig','first_payment','report','origination_year','ORIGINATION_COHORT','acquisition_file','primary','strict','reason','missing_months','unknown_active','repurchase','terminal_timing_disagreements','origination_field_conflict_count']].copy(),x],axis=1)
output['LEGACY_MIN_FICO_PROXY']=legacy; output['ORIGINATION_CLASSIC_FICO']=classic; output['ORIGINATION_VANTAGESCORE']=valid_number(first.vantage_score,300,850)
output['MI_MAPPING_INCONSISTENT']=mi_bad
output['half_year']=output.ORIGINATION_COHORT.str[:4]+'-H'+output.ORIGINATION_COHORT.str[-1].map({'1':'1','2':'1','3':'2','4':'2'})
for policy in ['primary','strict']:
    for name,flag in [('COMPOSITE',first.serious|first.adverse),('NUMERIC_90_PLUS',first.serious)]:
        output[f'TARGET_{policy.upper()}_{name}']=flag.astype('Int8').where(first[policy],pd.NA)
assert output.loan_id.is_unique and (output.strict<=output.primary).all()
assert output.loc[output.strict,'TARGET_PRIMARY_COMPOSITE'].equals(output.loc[output.strict,'TARGET_STRICT_COMPOSITE'])


## 5. Export and review
Produce one aggregate review ZIP without loan IDs or individual records. Event counts are descriptive; adequacy of half-year estimates must be assessed before Notebook 19. The final local Parquet is not approved for scoring until the review is accepted.

In [ ]:
run=OUT/'final'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); run.mkdir(parents=True)
local=run/'fannie_2015_2023_sampled_dataset.parquet'; output.to_parquet(local,index=False)
assert len(pd.read_parquet(local,columns=['loan_id']))==len(output)
summary=[]
for group in ['origination_year','half_year','ORIGINATION_COHORT']:
    for cohort,g in output.groupby(group):
        for policy in ['primary','strict']:
            target=g[f'TARGET_{policy.upper()}_COMPOSITE']; eligible=int(target.notna().sum()); events=int(target.sum())
            summary.append(dict(granularity=group,cohort=cohort,policy=policy,sampled=len(g),eligible=eligible,events=events,event_rate=events/eligible if eligible else None))
summary=pd.DataFrame(summary); summary.to_csv(run/'cohort_summary.csv',index=False)
pd.DataFrame([{k:v for k,v in m.items() if k not in ['counts','candidate_target_errors']} for m in metadata]).to_csv(run/'archive_inventory.csv',index=False)
pd.DataFrame([(m['archive'],k,v) for m in metadata for k,v in m['counts'].items()],columns=['archive','cohort_or_frame','loans']).to_csv(run/'acquisition_coverage.csv',index=False)
output.groupby(['origination_year','reason']).size().rename('loans').reset_index().to_csv(run/'eligibility_reasons.csv',index=False)
profiles=[]
for y,g in output.groupby('origination_year'):
    for c in list(x.columns)+['LEGACY_MIN_FICO_PROXY','ORIGINATION_CLASSIC_FICO','ORIGINATION_VANTAGESCORE']:
        profiles.append(dict(year=y,feature=c,missing=int(g[c].isna().sum()),distinct=int(g[c].nunique())))
pd.DataFrame(profiles).to_csv(run/'feature_profiles.csv',index=False)
manifest=dict(version=VERSION,years=YEARS,annual_sample=K,seed=SEED,sampling='lowest seeded BLAKE2b 64-bit hash of unique loan ID; before target eligibility',frame='acquisition year equal to origination year or following year; resolved origination dates only',dataset_sha256=sha(local),rows=len(output),primary_features=list(x.columns),fico_sensitivity_approved=False,release_label='release_2026Q2; user-provided folder label, verify release evidence',approval='pending_review',archive_count=len(metadata))
(run/'manifest.json').write_text(json.dumps(manifest,indent=2))
review=run/'notebook_18d_consolidated_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in run.iterdir():
        if p.suffix in {'.csv','.json'}: z.write(p,p.name)
display(summary.loc[summary.granularity.eq('half_year') & summary.policy.eq('primary')])
print('Local dataset:',local); print('Upload only:',review)
